In [2]:
import os
import random

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer
)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
import transformers

print("Transformers version:", transformers.__version__)

Transformers version: 5.17.0


In [4]:
DATA_PATH = "../data/processed/m_absa_selected.csv"

df = pd.read_csv(
    DATA_PATH,
    encoding="utf-8"
)

print("Shape:", df.shape)

df.head()

,text,aspect,category,sentiment,domain,language,split,text_length,word_count
0,We have gone for dinner only a few times but t...,service,service general,positive,restaurant,en,train,90,19
1,We have gone for dinner only a few times but t...,dinner,food quality,positive,restaurant,en,train,90,19
2,"Its dark , and cozy . . there is always jazz m...",NaN,ambience general,positive,restaurant,en,train,71,17
3,This place has great indian chinese food .,indian chinese food,food quality,positive,restaurant,en,train,42,8
4,Not what I would expect for the price and pres...,location,restaurant prices,neutral,restaurant,en,train,69,14


In [5]:
print("Sentiments:")
print(sorted(df["sentiment"].unique()))

print("\nNumber of categories:")
print(df["category"].nunique())

print("\nLanguages:")
print(sorted(df["language"].unique()))

print("\nDomains:")
print(sorted(df["domain"].unique()))

Sentiments:
['negative', 'neutral', 'positive']

Number of categories:
153

Languages:
['de', 'en', 'ru']

Domains:
['hotel', 'laptop', 'restaurant']


In [6]:
sentiment_labels = sorted(
    df["sentiment"].unique()
)

category_labels = sorted(
    df["category"].unique()
)

sentiment2id = {
    label: idx
    for idx, label in enumerate(sentiment_labels)
}

id2sentiment = {
    idx: label
    for label, idx in sentiment2id.items()
}

category2id = {
    label: idx
    for idx, label in enumerate(category_labels)
}

id2category = {
    idx: label
    for label, idx in category2id.items()
}

print("Sentiment mapping:")
print(sentiment2id)

print("\nNumber of category labels:", len(category2id))

Sentiment mapping:
{'negative': 0, 'neutral': 1, 'positive': 2}

Number of category labels: 153


In [7]:
# Проверяем основные поля

print("Missing values:")
print(
    df[
        ["text", "aspect", "sentiment", "category", "language", "domain", "split"]
    ].isna().sum()
)

print("\nRows:", len(df))

Missing values:
text            0
aspect       6234
sentiment       0
category        0
language        0
domain          0
split           0
dtype: int64

Rows: 27582


In [8]:
from transformers import AutoTokenizer

MODEL_NAME = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer:", MODEL_NAME)
print("Vocab size:", tokenizer.vocab_size)

Tokenizer: xlm-roberta-base
Vocab size: 250002


In [9]:
sample_text = str(df.iloc[0]["text"])
sample_aspect = str(df.iloc[0]["aspect"])

tokens = tokenizer(
    sample_text,
    sample_aspect,
    truncation=True,
    max_length=128
)

print("Text:")
print(sample_text)

print("\nAspect:")
print(sample_aspect)

print("\nTokens:")
print(tokenizer.convert_ids_to_tokens(tokens["input_ids"][:30]))

print("\nNumber of tokens:")
print(len(tokens["input_ids"]))

Text:
We have gone for dinner only a few times but the same great quality and service is given .

Aspect:
service

Tokens:
['<s>', '▁We', '▁have', '▁gone', '▁for', '▁dinner', '▁only', '▁a', '▁few', '▁times', '▁but', '▁the', '▁same', '▁great', '▁quality', '▁and', '▁service', '▁is', '▁given', '▁', '.', '</s>', '</s>', '▁service', '</s>']

Number of tokens:
25


In [10]:
# Разделение на train / validation / test

train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nTotal:", len(train_df) + len(val_df) + len(test_df))

Train: 16058
Validation: 3913
Test: 7611

Total: 27582


In [11]:
    # Статистика длины токенов

sample_size = min(5000, len(train_df))

sample_train = train_df.sample(
    n=sample_size,
    random_state=SEED
)

token_lengths = []

for _, row in sample_train.iterrows():

    text = str(row["text"])
    aspect = str(row["aspect"])

    encoded = tokenizer(
        text,
        aspect,
        truncation=False
    )

    token_lengths.append(
        len(encoded["input_ids"])
    )

token_lengths = np.array(token_lengths)

print("Количество samples:", len(token_lengths))
print("Min:", token_lengths.min())
print("Median:", np.median(token_lengths))
print("Mean:", token_lengths.mean())
print("95 percentile:", np.percentile(token_lengths, 95))
print("99 percentile:", np.percentile(token_lengths, 99))
print("Max:", token_lengths.max())

Количество samples: 5000
Min: 6
Median: 25.0
Mean: 28.5998
95 percentile: 58.0
99 percentile: 89.0
Max: 148


In [12]:
MAX_LENGTH = 128

print("MAX_LENGTH =", MAX_LENGTH)

MAX_LENGTH = 128


In [13]:


# Label mappings
sentiment_labels = sorted(df["sentiment"].unique())
category_labels = sorted(df["category"].unique())

sentiment2id = {
    label: idx
    for idx, label in enumerate(sentiment_labels)
}

id2sentiment = {
    idx: label
    for label, idx in sentiment2id.items()
}

category2id = {
    label: idx
    for idx, label in enumerate(category_labels)
}

id2category = {
    idx: label
    for label, idx in category2id.items()
}

# Numeric labels
df["sentiment_label"] = df["sentiment"].map(sentiment2id)
df["category_label"] = df["category"].map(category2id)

# Проверка
print("Sentiment classes:", sentiment_labels)
print("Number of sentiment classes:", len(sentiment_labels))

print("\nNumber of category classes:", len(category_labels))

print("\nMissing sentiment labels:", df["sentiment_label"].isna().sum())
print("Missing category labels:", df["category_label"].isna().sum())


# Train / validation / test
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "validation"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)

print("\nDataset sizes:")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

Sentiment classes: ['negative', 'neutral', 'positive']
Number of sentiment classes: 3

Number of category classes: 153

Missing sentiment labels: 0
Missing category labels: 0

Dataset sizes:
Train: 16058
Validation: 3913
Test: 7611
Total: 27582


In [14]:

# Hugging Face Datasets

from datasets import Dataset

columns = [
    "text",
    "aspect",
    "sentiment_label",
    "category_label"
]

train_dataset = Dataset.from_pandas(
    train_df[columns],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[columns],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[columns],
    preserve_index=False
)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'aspect', 'sentiment_label', 'category_label'],
    num_rows: 16058
})
Dataset({
    features: ['text', 'aspect', 'sentiment_label', 'category_label'],
    num_rows: 3913
})
Dataset({
    features: ['text', 'aspect', 'sentiment_label', 'category_label'],
    num_rows: 7611
})


In [15]:

# Создание Dataset + Tokenization

from datasets import Dataset

# ---------- 1. Подготовка pandas ----------

dataset_columns = [
    "text",
    "aspect",
    "sentiment_label",
    "category_label"
]

# Создаём копии
train_data = train_df[dataset_columns].copy()
val_data = val_df[dataset_columns].copy()
test_data = test_df[dataset_columns].copy()


# ---------- 2. Очистка text / aspect ----------

train_data["text"] = train_data["text"].fillna("").astype(str)
val_data["text"] = val_data["text"].fillna("").astype(str)
test_data["text"] = test_data["text"].fillna("").astype(str)

# NULL aspect оставляем как валидное значение
train_data["aspect"] = train_data["aspect"].apply(
    lambda x: "NULL" if pd.isna(x) else str(x)
)

val_data["aspect"] = val_data["aspect"].apply(
    lambda x: "NULL" if pd.isna(x) else str(x)
)

test_data["aspect"] = test_data["aspect"].apply(
    lambda x: "NULL" if pd.isna(x) else str(x)
)


# ---------- 3. Hugging Face Dataset ----------

train_dataset = Dataset.from_pandas(
    train_data,
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_data,
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_data,
    preserve_index=False
)


# ---------- 4. Tokenization ----------

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        examples["aspect"],
        truncation=True,
        max_length=MAX_LENGTH
    )


train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)


# ---------- 5. Переименовываем labels ----------

train_dataset = train_dataset.rename_column(
    "sentiment_label",
    "sentiment_labels"
)

train_dataset = train_dataset.rename_column(
    "category_label",
    "category_labels"
)

val_dataset = val_dataset.rename_column(
    "sentiment_label",
    "sentiment_labels"
)

val_dataset = val_dataset.rename_column(
    "category_label",
    "category_labels"
)

test_dataset = test_dataset.rename_column(
    "sentiment_label",
    "sentiment_labels"
)

test_dataset = test_dataset.rename_column(
    "category_label",
    "category_labels"
)


# ---------- 6. Проверка ----------

print("Train:", train_dataset)
print("Validation:", val_dataset)
print("Test:", test_dataset)

print("\nFeatures:")
print(train_dataset.features)

In [16]:

# Проверка токенизированного Dataset


example = train_dataset[0]

print("TEXT:")
print(example["text"])

print("\nASPECT:")
print(example["aspect"])

print("\nSENTIMENT LABEL:")
print(example["sentiment_labels"])

print("\nCATEGORY LABEL:")
print(example["category_labels"])

print("\nINPUT IDS (first 20):")
print(example["input_ids"][:20])

print("\nATTENTION MASK (first 20):")
print(example["attention_mask"][:20])

print("\nTOKENIZED TEXT:")
print(
    tokenizer.decode(
        example["input_ids"],
        skip_special_tokens=False
    )
)

TEXT:
We have gone for dinner only a few times but the same great quality and service is given .

ASPECT:
service

SENTIMENT LABEL:
2

CATEGORY LABEL:
152

INPUT IDS (first 20):
[0, 1401, 765, 74955, 100, 94000, 4734, 10, 10846, 20028, 1284, 70, 5701, 6782, 31897, 136, 4516, 83, 34475, 6]

ATTENTION MASK (first 20):
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

TOKENIZED TEXT:
<s> We have gone for dinner only a few times but the same great quality and service is given .</s></s> service</s>


In [17]:

# MultiTask XLM-R


import torch
import torch.nn as nn
from transformers import AutoModel


class MultiTaskXLMR(nn.Module):

    def __init__(
        self,
        model_name,
        num_sentiment_classes,
        num_category_classes,
        dropout=0.1
    ):
        super().__init__()

        # Shared XLM-R encoder
        self.encoder = AutoModel.from_pretrained(model_name)

        hidden_size = self.encoder.config.hidden_size

        # Shared dropout
        self.dropout = nn.Dropout(dropout)

        # Sentiment classification head
        self.sentiment_classifier = nn.Linear(
            hidden_size,
            num_sentiment_classes
        )

        # Category classification head
        self.category_classifier = nn.Linear(
            hidden_size,
            num_category_classes
        )

        # Loss functions
        self.sentiment_loss_fn = nn.CrossEntropyLoss()
        self.category_loss_fn = nn.CrossEntropyLoss()


    def forward(
        self,
        input_ids,
        attention_mask,
        sentiment_labels=None,
        category_labels=None
    ):

        # XLM-R encoder
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # <s> token representation
        pooled_output = outputs.last_hidden_state[:, 0, :]

        pooled_output = self.dropout(
            pooled_output
        )

        # Two prediction heads
        sentiment_logits = self.sentiment_classifier(
            pooled_output
        )

        category_logits = self.category_classifier(
            pooled_output
        )

        # Calculate total loss
        loss = None

        if sentiment_labels is not None and category_labels is not None:

            sentiment_loss = self.sentiment_loss_fn(
                sentiment_logits,
                sentiment_labels
            )

            category_loss = self.category_loss_fn(
                category_logits,
                category_labels
            )

            loss = sentiment_loss + category_loss

        return {
            "loss": loss,
            "sentiment_logits": sentiment_logits,
            "category_logits": category_logits
        }

In [18]:

# Create model
NUM_SENTIMENT_CLASSES = len(sentiment_labels)
NUM_CATEGORY_CLASSES = len(category_labels)

model = MultiTaskXLMR(
    model_name=MODEL_NAME,
    num_sentiment_classes=NUM_SENTIMENT_CLASSES,
    num_category_classes=NUM_CATEGORY_CLASSES
)

print("Model:", MODEL_NAME)
print("Sentiment classes:", NUM_SENTIMENT_CLASSES)
print("Category classes:", NUM_CATEGORY_CLASSES)
print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)

Model: xlm-roberta-base
Sentiment classes: 3
Category classes: 153
Parameters: 278163612


In [19]:

# Data Collator
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

print("Data collator:", type(data_collator).__name__)

Data collator: DataCollatorWithPadding


In [20]:

# Metrics

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)


def compute_metrics_from_predictions(
    sentiment_logits,
    category_logits,
    sentiment_labels,
    category_labels
):

    # Predictions
    sentiment_preds = np.argmax(
        sentiment_logits,
        axis=1
    )

    category_preds = np.argmax(
        category_logits,
        axis=1
    )

    # Sentiment metrics
    sentiment_precision, sentiment_recall, sentiment_f1, _ = (
        precision_recall_fscore_support(
            sentiment_labels,
            sentiment_preds,
            average="macro",
            zero_division=0
        )
    )

    sentiment_accuracy = accuracy_score(
        sentiment_labels,
        sentiment_preds
    )

    # Category metrics
    category_precision, category_recall, category_f1, _ = (
        precision_recall_fscore_support(
            category_labels,
            category_preds,
            average="macro",
            zero_division=0
        )
    )

    category_accuracy = accuracy_score(
        category_labels,
        category_preds
    )

    return {
        "sentiment_accuracy": sentiment_accuracy,
        "sentiment_precision_macro": sentiment_precision,
        "sentiment_recall_macro": sentiment_recall,
        "sentiment_f1_macro": sentiment_f1,

        "category_accuracy": category_accuracy,
        "category_precision_macro": category_precision,
        "category_recall_macro": category_recall,
        "category_f1_macro": category_f1,
    }

In [21]:

# Custom Trainer for MultiTask XLM-R


from transformers import Trainer


class ABSA_Trainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        sentiment_labels = inputs.pop("sentiment_labels")
        category_labels = inputs.pop("category_labels")

        outputs = model(
            **inputs,
            sentiment_labels=sentiment_labels,
            category_labels=category_labels
        )

        loss = outputs["loss"]

        if return_outputs:
            return loss, outputs

        return loss


    def prediction_step(
        self,
        model,
        inputs,
        prediction_loss_only,
        ignore_keys=None
    ):
        """
        Custom prediction step for two classification heads.
        """

        has_labels = (
            "sentiment_labels" in inputs
            and "category_labels" in inputs
        )

        inputs = self._prepare_inputs(inputs)

        if has_labels:
            sentiment_labels = inputs.pop("sentiment_labels")
            category_labels = inputs.pop("category_labels")
        else:
            sentiment_labels = None
            category_labels = None

        with torch.no_grad():

            outputs = model(
                **inputs,
                sentiment_labels=sentiment_labels,
                category_labels=category_labels
            )

        loss = outputs["loss"] if has_labels else None

        if prediction_loss_only:
            return loss, None, None

        logits = (
            outputs["sentiment_logits"],
            outputs["category_logits"]
        )

        labels = None

        if has_labels:
            labels = (
                sentiment_labels,
                category_labels
            )

        return loss, logits, labels

In [22]:

# Forward pass test


# Берём 4 примера
examples = [
    train_dataset[i]
    for i in range(4)
]

# Оставляем только нужные для модели поля
features = [
    {
        "input_ids": example["input_ids"],
        "attention_mask": example["attention_mask"],
        "sentiment_labels": example["sentiment_labels"],
        "category_labels": example["category_labels"],
    }
    for example in examples
]

# Padding
batch = data_collator(features)

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.eval()

# Переносим tensors
batch_inputs = {
    key: value.to(device)
    for key, value in batch.items()
}

# Forward pass
with torch.no_grad():
    outputs = model(**batch_inputs)

print("Device:", device)

print("Loss:", outputs["loss"].item())

print(
    "Sentiment logits shape:",
    outputs["sentiment_logits"].shape
)

print(
    "Category logits shape:",
    outputs["category_logits"].shape
)

Device: cuda
Loss: 6.442075252532959
Sentiment logits shape: torch.Size([4, 3])
Category logits shape: torch.Size([4, 153])


In [23]:

# Training configuration

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="../models/absa/xlmr_multitask",

    # Training
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,

    # Batch
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,

    # Evaluation
    eval_strategy="epoch",
    save_strategy="epoch",

    # Best model
    load_best_model_at_end=True,
    metric_for_best_model="eval_sentiment_f1_macro",
    greater_is_better=True,

    # Performance
    fp16=torch.cuda.is_available(),

    # Logging
    logging_strategy="steps",
    logging_steps=100,

    # Reproducibility
    seed=SEED,

    # Dataset
    remove_unused_columns=False,

    # Save
    save_total_limit=2,

    report_to="none"
)

print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start

In [24]:
train_dataset = train_dataset.remove_columns(["text", "aspect"])
val_dataset = val_dataset.remove_columns(["text", "aspect"])
test_dataset = test_dataset.remove_columns(["text", "aspect"])

print(train_dataset.features)

In [25]:
def compute_metrics(eval_pred):
    sentiment_logits, category_logits = eval_pred.predictions
    sentiment_labels, category_labels = eval_pred.label_ids

    return compute_metrics_from_predictions(
        sentiment_logits,
        category_logits,
        sentiment_labels,
        category_labels
    )

In [26]:
trainer = ABSA_Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer created successfully")

Trainer created successfully


In [27]:
train_loader = trainer.get_train_dataloader()

batch = next(iter(train_loader))

print("Batch keys:", batch.keys())

for key, value in batch.items():
    print(
        key,
        "| shape:", value.shape,
        "| dtype:", value.dtype
    )

Batch keys: KeysView({'sentiment_labels': tensor([2, 2, 2, 2, 2, 0, 2, 2], device='cuda:0'), 'category_labels': tensor([ 47, 121, 152,  48, 120,  54, 130,  38], device='cuda:0'), 'input_ids': tensor([[     0,   6117,     19,      6,      4,  22729,      6,      4,  21185,
            621,   2367,     17,  12319,   3229, 104064,    442,    136,  85358,
          10932,      6,      5,      2,      2,  18057,  23708,      2,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1],
        [     0,   1858, 119482,    319,   4834,      9,  73347,  33657,   1631,
         170786,    165,    654,  20244,   2242,   7057,    198,      5,      2,
              2, 119482,    319,   4834,      9,  73347,  33657,      2,      1,
              1,      1,      1,      1,      1,      1,      1,      1,      1,
              1,      1,      1,      1],
        [     0,   1858, 114667,  17915,    443,  14573, 139599,   1146,    

In [28]:
model.train()

batch = {
    key: value.to(device)
    for key, value in batch.items()
}

with torch.no_grad():
    outputs = model(**batch)

print("Loss:", outputs["loss"].item())
print("Sentiment logits:", outputs["sentiment_logits"].shape)
print("Category logits:", outputs["category_logits"].shape)

Loss: 6.269393444061279
Sentiment logits: torch.Size([8, 3])
Category logits: torch.Size([8, 153])


In [29]:
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Sentiment Accuracy,Sentiment Precision Macro,Sentiment Recall Macro,Sentiment F1 Macro,Category Accuracy,Category Precision Macro,Category Recall Macro,Category F1 Macro
1,2.103428,2.081042,0.891899,0.580564,0.615843,0.597593,0.575773,0.118951,0.141895,0.112439
2,1.513202,1.801140,0.893432,0.676837,0.619030,0.612764,0.624840,0.166694,0.199837,0.165460
3,1.371221,1.768949,0.897777,0.750400,0.634578,0.637416,0.626374,0.180440,0.213165,0.176955


In [30]:
import os
import torch

MODEL_DIR = "../models/absa/xlmr_multitask"

os.makedirs(MODEL_DIR, exist_ok=True)

torch.save(
    model.state_dict(),
    f"{MODEL_DIR}/model.pt"
)

tokenizer.save_pretrained(MODEL_DIR)

print("Model saved to:", MODEL_DIR)

Model saved to: ../models/absa/xlmr_multitask


In [31]:
import os

print(os.path.exists("../models/absa/xlmr_multitask/model.pt"))
print(os.path.getsize("../models/absa/xlmr_multitask/model.pt") / 1024**2, "MB")

True
1061.177435874939 MB


In [32]:
test_metrics = trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test"
)

test_metrics

{'test_loss': 1.767113447189331,
 'test_sentiment_accuracy': 0.9013270266719222,
 'test_sentiment_precision_macro': 0.6619966649919596,
 'test_sentiment_recall_macro': 0.6077185704601772,
 'test_sentiment_f1_macro': 0.6047003208879137,
 'test_category_accuracy': 0.62238864800946,
 'test_category_precision_macro': 0.1798878885528154,
 'test_category_recall_macro': 0.19622902527351044,
 'test_category_f1_macro': 0.17366298621249973}

In [33]:
import pandas as pd

transformer_results = pd.DataFrame([test_metrics])

transformer_results.to_csv(
    "../data/processed/transformer_test_results.csv",
    index=False
)

transformer_results

,test_loss,test_sentiment_accuracy,test_sentiment_precision_macro,test_sentiment_recall_macro,test_sentiment_f1_macro,test_category_accuracy,test_category_precision_macro,test_category_recall_macro,test_category_f1_macro
0,1.767113,0.901327,0.661997,0.607719,0.6047,0.622389,0.179888,0.196229,0.173663
